In [1]:
import os

- The token is stored on 'AI_API_KEY' variable.

In [ ]:
os.environ['AI_API_KEY'] = '<insert_token>'

In [4]:
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["AI_API_KEY"],
)

In [5]:
completion = client.chat.completions.create(
    model="zai-org/GLM-5.3-Flash:fireworks-ai",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe the emotions that can be obtained from the following text."
                },
                {
                    "type": "text",
                    "text": """Markdown is a simple way to write and format text using plain text symbols. It's a lightweight language that lets you create nicely formatted text without the use of complex coding. The idea is to make it easy for people to read and understand the formatting, even if they're looking at the raw text.
                    
                    This cheat sheet will go over some of the key components of markdown, including code syntaxes and examples that the reader can use to create their own highly formatted text articles."""
                }
            ]
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The text is primarily informational, so it evokes mostly subtle, low-intensity emotions rather than strong feelings. Here are the emotions a reader might experience:\n\n**1. Reassurance / Ease**\n- Phrases like *"simple way," "lightweight language," "without the use of complex coding,"* and *"easy for people to read and understand"* are deliberately calming. They lower the reader\'s anxiety about tackling something technical and create a sense that this is approachable for anyone.\n\n**2. Curiosity / Interest**\n- The mention of a *"cheat sheet"* covering *"key components... syntaxes and examples"* prompts the reader to wonder what they\'ll learn, encouraging them to keep reading.\n\n**3. Confidence / Empowerment**\n- The promise that readers can *"create their own highly formatted text articles"* instills a sense of capability — the feeling that this skill is within their grasp.\n\n**4. Welcoming / Inclusiveness**\n- The emphasis on making content readab

In [6]:
print(completion.choices[0].message.content)

The text is primarily informational, so it evokes mostly subtle, low-intensity emotions rather than strong feelings. Here are the emotions a reader might experience:

**1. Reassurance / Ease**
- Phrases like *"simple way," "lightweight language," "without the use of complex coding,"* and *"easy for people to read and understand"* are deliberately calming. They lower the reader's anxiety about tackling something technical and create a sense that this is approachable for anyone.

**2. Curiosity / Interest**
- The mention of a *"cheat sheet"* covering *"key components... syntaxes and examples"* prompts the reader to wonder what they'll learn, encouraging them to keep reading.

**3. Confidence / Empowerment**
- The promise that readers can *"create their own highly formatted text articles"* instills a sense of capability — the feeling that this skill is within their grasp.

**4. Welcoming / Inclusiveness**
- The emphasis on making content readable *"even if they're looking at the raw text"

## Natural Language to SQL

In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('Dataset/sales_data_sample.csv')

In [ ]:
# Checking the data in df
print(df.info())
print(df.head(5))

<class 'pandas.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2823 non-null   int64  
 1   QUANTITYORDERED   2823 non-null   int64  
 2   PRICEEACH         2823 non-null   float64
 3   SALES             2823 non-null   float64
 4   ORDERDATE         2823 non-null   str    
 5   QTR_ID            2823 non-null   int64  
 6   MONTH_ID          2823 non-null   int64  
 7   YEAR_ID           2823 non-null   int64  
 8   PRODUCTLINE       2823 non-null   str    
 9   PHONE             2823 non-null   str    
 10  ADDRESSLINE1      2823 non-null   str    
 11  CITY              2823 non-null   str    
 12  STATE             1337 non-null   str    
 13  POSTALCODE        2747 non-null   str    
 14  COUNTRY           2823 non-null   str    
 15  CONTACTLASTNAME   2823 non-null   str    
 16  CONTACTFIRSTNAME  2823 non-null   str    
dtypes: flo

In [ ]:
# Querying for the sum of the sales in each quarter
print(df.groupby("QTR_ID").sum()['SALES'])

QTR_ID
1    2350817.73
2    2048120.30
3    1758910.81
4    3874780.01
Name: SALES, dtype: float64


In [11]:
from sqlalchemy import create_engine
from sqlalchemy import text

- Creating a temporary database in memory and coping the df data into it.
- This can be queried using SQL Language.

In [12]:
temp_db = create_engine('sqlite:///:memory:', echo=True)

**NOTE:** sqlalchemy can be used to connect to actual databases like MySQL and PostgreSQL.

In [13]:
data = df.to_sql(name='Sales', con=temp_db)

2026-09-16 16:53:38,258 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-16 16:53:38,270 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("Sales")
2026-09-16 16:53:38,272 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-16 16:53:38,274 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("Sales")
2026-09-16 16:53:38,275 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-16 16:53:38,279 INFO sqlalchemy.engine.Engine 
CREATE TABLE "Sales" (
	"index" BIGINT, 
	"ORDERNUMBER" BIGINT, 
	"QUANTITYORDERED" BIGINT, 
	"PRICEEACH" FLOAT, 
	"SALES" FLOAT, 
	"ORDERDATE" TEXT, 
	"QTR_ID" BIGINT, 
	"MONTH_ID" BIGINT, 
	"YEAR_ID" BIGINT, 
	"PRODUCTLINE" TEXT, 
	"PHONE" TEXT, 
	"ADDRESSLINE1" TEXT, 
	"CITY" TEXT, 
	"STATE" TEXT, 
	"POSTALCODE" TEXT, 
	"COUNTRY" TEXT, 
	"CONTACTLASTNAME" TEXT, 
	"CONTACTFIRSTNAME" TEXT
)


2026-09-16 16:53:38,280 INFO sqlalchemy.engine.Engine [no key 0.00128s] ()
2026-09-16 16:53:38,284 INFO sqlalchemy.engine.Engine CREATE INDEX "ix_Sales_index" ON "Sales" (

In [24]:
# Make a connection to run code and close it
with temp_db.connect() as conn:
  result = conn.execute(text("SELECT * FROM Sales WHERE QTR_ID = 3 AND YEAR_ID = 2003 AND PRODUCTLINE = 'Motorcycles'"))
  sum_result = conn.execute(text("SELECT QTR_ID, SUM(SALES) FROM Sales GROUP BY QTR_ID"))

2026-09-16 17:15:17,076 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-16 17:15:17,078 INFO sqlalchemy.engine.Engine SELECT * FROM Sales WHERE QTR_ID = 3 AND YEAR_ID = 2003 AND PRODUCTLINE = 'Motorcycles'
2026-09-16 17:15:17,080 INFO sqlalchemy.engine.Engine [cached since 218.8s ago] ()
2026-09-16 17:15:17,082 INFO sqlalchemy.engine.Engine SELECT QTR_ID, SUM(SALES) FROM Sales GROUP BY QTR_ID
2026-09-16 17:15:17,084 INFO sqlalchemy.engine.Engine [generated in 0.00165s] ()
2026-09-16 17:15:17,086 INFO sqlalchemy.engine.Engine ROLLBACK


In [21]:
result.all()

[(2, 10134, 41, 94.74, 3884.34, '7/1/2003 0:00', 3, 7, 2003, 'Motorcycles', '+33 1 46 62 7555', '27 rue du Colonel Pierre Avia', 'Paris', None, '75508', 'France', 'Da Cunha', 'Daniel'),
 (3, 10145, 45, 83.26, 3746.7, '8/25/2003 0:00', 3, 8, 2003, 'Motorcycles', '6265557265', '78934 Hillside Dr.', 'Pasadena', 'CA', '90003', 'USA', 'Young', 'Julie'),
 (56, 10134, 27, 100.0, 3307.77, '7/1/2003 0:00', 3, 7, 2003, 'Motorcycles', '+33 1 46 62 7555', '27 rue du Colonel Pierre Avia', 'Paris', None, '75508', 'France', 'Da Cunha', 'Daniel'),
 (57, 10145, 37, 100.0, 5192.95, '8/25/2003 0:00', 3, 8, 2003, 'Motorcycles', '6265557265', '78934 Hillside Dr.', 'Pasadena', 'CA', '90003', 'USA', 'Young', 'Julie'),
 (82, 10134, 31, 100.0, 7023.98, '7/1/2003 0:00', 3, 7, 2003, 'Motorcycles', '+33 1 46 62 7555', '27 rue du Colonel Pierre Avia', 'Paris', None, '75508', 'France', 'Da Cunha', 'Daniel'),
 (83, 10145, 33, 100.0, 5176.38, '8/25/2003 0:00', 3, 8, 2003, 'Motorcycles', '6265557265', '78934 Hillside 

In [25]:
sum_result.all()

[(1, 2350817.73), (2, 2048120.3), (3, 1758910.81), (4, 3874780.01)]

In [26]:
# Function to obtain the df defination - this for sqlite SQL
def get_table_definition(df):
  prompt = f"""### sqlite SQL table, with its properties:
  #
  # Sales({', '.join(str(col) for col in df.columns)})
  #
  """

  return prompt

In [27]:
#example
print(get_table_definition(df))

### sqlite SQL table, with its properties:
  #
  # Sales(ORDERNUMBER, QUANTITYORDERED, PRICEEACH, SALES, ORDERDATE, QTR_ID, MONTH_ID, YEAR_ID, PRODUCTLINE, PHONE, ADDRESSLINE1, CITY, STATE, POSTALCODE, COUNTRY, CONTACTLASTNAME, CONTACTFIRSTNAME)
  #
  


In [30]:
# Grabing the input from the user
def get_prompt_input():
  prompt_text = input("Enter the query to be performed: ")
  return prompt_text

In [32]:
def final_prompt(df, query):
  definition = get_table_definition(df)
  query_string = f"### A query to answer: {query}\n SELECT"
  return definition + query_string

In [33]:
query = get_prompt_input()
print(query)

prompt = final_prompt(df, query)

Grab all the sales per quarter.


In [34]:
print(prompt)

### sqlite SQL table, with its properties:
  #
  # Sales(ORDERNUMBER, QUANTITYORDERED, PRICEEACH, SALES, ORDERDATE, QTR_ID, MONTH_ID, YEAR_ID, PRODUCTLINE, PHONE, ADDRESSLINE1, CITY, STATE, POSTALCODE, COUNTRY, CONTACTLASTNAME, CONTACTFIRSTNAME)
  #
  ### A query to answer: Grab all the sales per quarter.
 SELECT


In [39]:
def gen_query(info):
  completion = client.chat.completions.create(
    model="zai-org/GLM-5.3-Flash:fireworks-ai",
    messages=[
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": "Given the following text, write an SQL query to satisfy the given query requirements. It should be a single statement as a python string and nothing else. Not in Markdown format."
          },
          {
            "type": "text",
            "text": info
          }
        ]
      }
    ],
  )
  return completion.choices[0].message.content

In [ ]:
# Using the generated prompt:
sql_query = gen_query(prompt)
print(sql_query)

"SELECT QTR_ID, SUM(SALES) AS TOTAL_SALES FROM Sales GROUP BY QTR_ID"


In [44]:
# Query function
def perform_query(query):
  with temp_db.connect() as conn:
    result = conn.execute(text(query.replace('"', "")))
  return result

In [46]:
# Using the generated query
print(perform_query(sql_query).all())

2026-09-16 17:56:15,957 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-16 17:56:15,958 INFO sqlalchemy.engine.Engine SELECT QTR_ID, SUM(SALES) AS TOTAL_SALES FROM Sales GROUP BY QTR_ID
2026-09-16 17:56:15,959 INFO sqlalchemy.engine.Engine [cached since 15.88s ago] ()
2026-09-16 17:56:15,960 INFO sqlalchemy.engine.Engine ROLLBACK
[(1, 2350817.73), (2, 2048120.3), (3, 1758910.81), (4, 3874780.01)]
